# Feature Engineering


In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from market_ml.data_loader import DataLoader
import matplotlib.pyplot as plt
import plotly.express as px
from market_ml.config import * 
from market_ml.EDA import * 
from market_ml.feature_engineering import *

# Data Loading 


In [2]:
stock_dl = DataLoader(tickers=TICKERS, period="10y", interval="1d", file_name='stock_data.parquet')
benchmark_dl = DataLoader(tickers=[BENCHMARK_TICKERS], period="10y", interval="1d", file_name='benchmark_data.parquet')

market_data = stock_dl.load_data()
benchmark_data = benchmark_dl.load_data()
sectors = market_data["Sector"].unique()

benchmark_data

Data loaded for ['AAPL', 'MSFT', 'NVDA', 'META', 'GOOGL', 'JPM', 'GS', 'BAC', 'MS', 'BLK', 'JNJ', 'UNH', 'ABBV', 'PFE', 'MRK', 'XOM', 'CVX', 'COP', 'SLB', 'EOG', 'AMZN', 'COST', 'WMT', 'HD', 'MCD']
Data loaded for [['SPY', 'QQQ']]


Price,Date,Ticker,Sector,Close,High,Low,Open,Volume
0,2016-09-29,QQQ,NaN,110.121559,110.943924,109.644969,110.738333,27207500
1,2016-09-29,SPY,NaN,183.335754,185.206007,182.789198,184.804628,128070600
2,2016-09-30,QQQ,NaN,110.943939,111.271013,110.299131,110.411274,25543200
3,2016-09-30,SPY,NaN,184.719223,185.419493,183.916465,184.164118,117202900
4,2016-10-03,QQQ,NaN,110.785042,110.925211,110.355165,110.803729,19216400
...,...,...,...,...,...,...,...,...
5021,2026-09-25,SPY,NaN,771.349976,772.280029,766.289978,768.780029,36666700
5022,2026-09-28,QQQ,NaN,736.530029,741.419983,731.630005,740.460022,41716200
5023,2026-09-28,SPY,NaN,765.609985,769.539978,763.719971,768.349976,42296700
5024,2026-09-29,QQQ,NaN,737.229980,740.579895,736.000000,740.169983,9703756


In [3]:
#just testing with one stock for now 

test_ticker = ['AAPL', 'NVDA']
test_data = market_data[market_data["Ticker"].isin(test_ticker)].copy()

test_data

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume
0,2016-09-29,AAPL,Technology,25.653965,26.024437,25.567065,25.878078,143548000
19,2016-09-29,NVDA,Technology,1.653514,1.658666,1.620394,1.635605,336684000
25,2016-09-30,AAPL,Technology,25.852909,25.926088,25.567052,25.717984,145516400
44,2016-09-30,NVDA,Technology,1.680990,1.697427,1.652778,1.662591,429932000
50,2016-10-03,AAPL,Technology,25.731701,25.852906,25.676817,25.775152,86807200
...,...,...,...,...,...,...,...,...
62769,2026-09-25,NVDA,Technology,225.070007,226.940002,223.130005,225.130005,89947700
62775,2026-09-28,AAPL,Technology,338.399994,342.989990,338.040009,340.369995,32724700
62794,2026-09-28,NVDA,Technology,228.860001,233.210007,228.039993,229.750000,141544300
62800,2026-09-29,AAPL,Technology,332.345001,337.059998,331.510010,337.059998,12025955


In [4]:
FE = FeatureEngineer(test_data, benchmark_data)



Benchmark Features

In [5]:
list(benchmark_data["Ticker"].unique())

['QQQ', 'SPY']

# Returns for the benchmark

In [6]:

test_data = FE.add_benchmark_returns()
test_data

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume,QQQ_Return,SPY_Return
0,2016-09-29,AAPL,Technology,25.653965,26.024437,25.567065,25.878078,143548000,NaN,NaN
1,2016-09-29,NVDA,Technology,1.653514,1.658666,1.620394,1.635605,336684000,NaN,NaN
2,2016-09-30,AAPL,Technology,25.852909,25.926088,25.567052,25.717984,145516400,0.007468,0.007546
3,2016-09-30,NVDA,Technology,1.680990,1.697427,1.652778,1.662591,429932000,0.007468,0.007546
4,2016-10-03,AAPL,Technology,25.731701,25.852906,25.676817,25.775152,86807200,-0.001432,-0.002404
...,...,...,...,...,...,...,...,...,...,...
5021,2026-09-25,NVDA,Technology,225.070007,226.940002,223.130005,225.130005,89947700,0.004588,0.005435
5022,2026-09-28,AAPL,Technology,338.399994,342.989990,338.040009,340.369995,32724700,-0.010705,-0.007441
5023,2026-09-28,NVDA,Technology,228.860001,233.210007,228.039993,229.750000,141544300,-0.010705,-0.007441
5024,2026-09-29,AAPL,Technology,332.345001,337.059998,331.510010,337.059998,12025955,0.000950,-0.002338


# Returns

In [7]:

test_data = FE.add_daily_returns() #Daily Price Movement 
test_data = FE.add_return_feature() #Very short term momentum - default is 5 days. 

# Volatility Features

In [8]:

test_data = FE.add_volatility_feature() #recent variability - default is 20 days

# Trend Features

In [9]:
test_data = FE.add_SMA_feature() #recent price trend - default is 20 
test_data = FE.add_SMA_feature(50) #longer term price trend 
test_data = FE.add_price_relative_to_sma() #position relative to trend - default is 20
test_data = FE.add_price_relative_to_sma(50) #position relative to longer trend 

test_data.tail(20)

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume,QQQ_Return,SPY_Return,Daily_Return,Return_5D,Volatility_20D,SMA_20D,SMA_50D,Price_to_SMA_20D,Price_to_SMA_50D
5006,2026-09-16,AAPL,Technology,332.410004,335.480011,330.700012,332.529999,35981000,0.000255,-0.004410,0.003229,0.054132,0.014942,320.389998,319.144035,1.037517,1.041567
5007,2026-09-16,NVDA,Technology,213.899994,216.759995,212.500000,214.139999,96563600,0.000255,-0.004410,0.008154,-0.042610,0.028648,218.334263,213.363682,0.979690,1.002514
5008,2026-09-17,AAPL,Technology,337.000000,338.339996,330.179993,334.769989,36700200,0.017312,0.011339,0.013808,0.031938,0.014522,321.398499,319.621636,1.048543,1.054372
5009,2026-09-17,NVDA,Technology,219.339996,219.910004,217.149994,218.380005,94191300,0.017312,0.011339,0.025432,0.004488,0.029149,218.435421,213.672645,1.004141,1.026524
5010,2026-09-18,AAPL,Technology,336.130005,338.489990,332.529999,337.910004,86588200,0.006319,0.001287,-0.002582,0.011617,0.013771,322.639999,320.025286,1.041811,1.050323
5011,2026-09-18,NVDA,Technology,222.270004,222.729996,218.029999,219.350006,190287400,0.006319,0.001287,0.013358,0.018233,0.029262,218.718539,214.066978,1.016238,1.038320
5012,2026-09-21,AAPL,Technology,338.980011,339.640015,333.049988,335.279999,34999200,0.028821,0.015505,0.008479,0.017714,0.013590,324.121500,320.503920,1.045842,1.057647
5013,2026-09-21,NVDA,Technology,227.380005,228.500000,221.559998,222.940002,109806100,0.028821,0.015505,0.022990,0.077835,0.029501,219.363539,214.400094,1.036544,1.060541
5014,2026-09-22,AAPL,Technology,339.750000,345.339996,338.750000,340.140015,40711800,0.008079,-0.000155,0.002271,0.025382,0.013596,325.592000,320.958188,1.043484,1.058549
5015,2026-09-22,NVDA,Technology,228.869995,229.979996,226.500000,226.850006,95746700,0.008079,-0.000155,0.006553,0.078710,0.028501,220.394690,214.911444,1.038455,1.064950


# Volume Features

In [10]:
test_data = FE.add_average_volume_feature() #Trading activity - default is 20

test_data = FE.add_relative_volume()

test_data

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume,QQQ_Return,SPY_Return,Daily_Return,Return_5D,Volatility_20D,SMA_20D,SMA_50D,Price_to_SMA_20D,Price_to_SMA_50D,Volume_Average_20D,Relative_Volume_20D
0,2016-09-29,AAPL,Technology,25.653965,26.024437,25.567065,25.878078,143548000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-09-29,NVDA,Technology,1.653514,1.658666,1.620394,1.635605,336684000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2016-09-30,AAPL,Technology,25.852909,25.926088,25.567052,25.717984,145516400,0.007468,0.007546,0.007755,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2016-09-30,NVDA,Technology,1.680990,1.697427,1.652778,1.662591,429932000,0.007468,0.007546,0.016617,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2016-10-03,AAPL,Technology,25.731701,25.852906,25.676817,25.775152,86807200,-0.001432,-0.002404,-0.004688,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5021,2026-09-25,NVDA,Technology,225.070007,226.940002,223.130005,225.130005,89947700,0.004588,0.005435,0.002182,0.012597,0.020290,221.654555,215.794765,1.015409,1.042982,1.160675e+08,0.774960
5022,2026-09-28,AAPL,Technology,338.399994,342.989990,338.040009,340.369995,32724700,-0.010705,-0.007441,-0.007828,-0.001711,0.014007,330.331000,321.841754,1.024427,1.051448,4.259910e+07,0.768202
5023,2026-09-28,NVDA,Technology,228.860001,233.210007,228.039993,229.750000,141544300,-0.010705,-0.007441,0.016839,0.006509,0.017570,222.232213,216.320299,1.029824,1.057968,1.133889e+08,1.248308
5024,2026-09-29,AAPL,Technology,332.345001,337.059998,331.510010,337.059998,12025955,0.000950,-0.002338,-0.017893,-0.021795,0.014541,331.105750,321.962483,1.003743,1.032248,4.113826e+07,0.292330


# Target Variable - 5 Day return

In [14]:
test_data = FE.add_target_variable(5)

test_data

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume,QQQ_Return,SPY_Return,...,Return_5D,Volatility_20D,SMA_20D,SMA_50D,Price_to_SMA_20D,Price_to_SMA_50D,Volume_Average_20D,Relative_Volume_20D,Future_Return_5D,Target_5D
0,2016-09-29,AAPL,Technology,25.653965,26.024437,25.567065,25.878078,143548000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.015243,1.0
1,2016-09-29,NVDA,Technology,1.653514,1.658666,1.620394,1.635605,336684000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.000890,0.0
2,2016-09-30,AAPL,Technology,25.852909,25.926088,25.567052,25.717984,145516400,0.007468,0.007546,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.008934,1.0
3,2016-09-30,NVDA,Technology,1.680990,1.697427,1.652778,1.662591,429932000,0.007468,0.007546,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.024373,0.0
4,2016-10-03,AAPL,Technology,25.731701,25.852906,25.676817,25.775152,86807200,-0.001432,-0.002404,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.031373,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5021,2026-09-25,NVDA,Technology,225.070007,226.940002,223.130005,225.130005,89947700,0.004588,0.005435,...,0.012597,0.020290,221.654555,215.794765,1.015409,1.042982,1.160675e+08,0.774960,NaN,NaN
5022,2026-09-28,AAPL,Technology,338.399994,342.989990,338.040009,340.369995,32724700,-0.010705,-0.007441,...,-0.001711,0.014007,330.331000,321.841754,1.024427,1.051448,4.259910e+07,0.768202,NaN,NaN
5023,2026-09-28,NVDA,Technology,228.860001,233.210007,228.039993,229.750000,141544300,-0.010705,-0.007441,...,0.006509,0.017570,222.232213,216.320299,1.029824,1.057968,1.133889e+08,1.248308,NaN,NaN
5024,2026-09-29,AAPL,Technology,332.345001,337.059998,331.510010,337.059998,12025955,0.000950,-0.002338,...,-0.021795,0.014541,331.105750,321.962483,1.003743,1.032248,4.113826e+07,0.292330,NaN,NaN


In [12]:
test_data = add_binary_target(test_data)

test_data.head(10)

NameError: name 'add_binary_target' is not defined

In [ ]:
test_data = FE.add_binary_target(horizon=5)

AttributeError: 'FeatureEngineer' object has no attribute 'add_binary_target'

In [ ]:
test_data.tail(50)


Price,Date,Ticker,Sector,Close,High,Low,Open,Volume,QQQ_Return,SPY_Return,...,Return_5D,Volatility_20D,SMA_20D,SMA_50D,Price_to_SMA_20D,Price_to_SMA_50D,Volume_Average_20D,Relative_Volume_20D,Target,Target_5D
4976,2026-08-25,AAPL,Technology,309.899994,313.589996,308.209991,310.790009,25869800,0.006229,0.003196,...,-0.000419,0.019989,311.391994,310.673482,0.995209,0.997510,5.097356e+07,0.507514,1,NaN
4977,2026-08-25,NVDA,Technology,212.811874,214.489989,209.875158,210.794128,122309000,0.006229,0.003196,...,-0.030445,0.022402,214.338165,207.574132,0.992879,1.025233,1.150328e+08,1.063253,1,NaN
4978,2026-08-26,AAPL,Technology,313.450012,315.429993,308.799988,310.299988,34024500,0.000915,0.000222,...,-0.010668,0.020296,310.169565,311.019191,1.010576,1.007816,4.987024e+07,0.682261,1,NaN
4979,2026-08-26,NVDA,Technology,209.425659,213.361258,208.996132,212.402324,179900000,0.000915,0.000222,...,-0.036312,0.020956,215.319567,207.518394,0.972627,1.009191,1.166438e+08,1.542302,1,NaN
4980,2026-08-27,AAPL,Technology,314.579987,315.399994,309.399994,310.549988,32419200,0.013692,0.006553,...,0.010536,0.020198,309.241431,311.331147,1.017263,1.010435,4.775031e+07,0.678932,1,NaN
4981,2026-08-27,NVDA,Technology,227.725174,230.212396,220.653086,222.610902,298909800,0.013692,0.006553,...,0.051326,0.027593,216.964726,207.929334,1.049595,1.095205,1.251388e+08,2.388627,1,NaN
4982,2026-08-28,AAPL,Technology,319.700012,322.369995,315.450012,316.850006,38649400,-0.006490,-0.002269,...,0.033457,0.011884,309.794240,311.811248,1.031975,1.025300,4.305832e+07,0.897606,1,NaN
4983,2026-08-28,NVDA,Technology,217.306839,229.003742,216.567661,227.105872,195116400,-0.006490,-0.002269,...,0.013180,0.029602,217.803787,208.187046,0.997718,1.043806,1.278965e+08,1.525580,1,NaN
4984,2026-08-31,AAPL,Technology,316.850006,321.239990,312.799988,319.600006,41242700,0.000461,-0.002990,...,0.020977,0.011264,310.478812,312.193184,1.020521,1.014916,4.136786e+07,0.996974,0,NaN
4985,2026-08-31,NVDA,Technology,220.533234,221.052657,215.968349,218.625365,124702700,0.000461,-0.002990,...,0.058998,0.029134,218.509997,208.388620,1.009259,1.058279,1.277113e+08,0.976442,1,NaN
